### Remove the beams 10 and 11

In [4]:
import os
import multiprocessing
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '6'
os.environ['MKL_NUM_THREADS'] = '6'
os.environ['OPENBLAS_NUM_THREADS'] = '6'
os.environ['NUMEXPR_NUM_THREADS'] = '6'
os.environ['VECLIB_MAXIMUM_THREADS'] = '6'
os.environ['BLAS_NUM_THREADS'] = '6'

# Force multiprocessing to respect thread limits
multiprocessing.set_start_method('spawn', force=True)
from Modules import pre_pro
import torch
from torch.utils.data import TensorDataset, DataLoader
from Modules.Neural_Net import Net
from Modules.Losses import crit
import numpy as np
from Modules.beams import beam_trim
import matplotlib.pyplot as plt

In [3]:
data = torch.load('../../data/data.pt', weights_only=True)
frames = np.arange(250,350, 1)
device = 'cuda'

beams = list(range(0, 32))
indices_to_remove = [10, 11]
beams = [b for i, b in enumerate(beams) if i not in indices_to_remove]

with torch.no_grad():
    input, output = pre_pro.in_out(data, beams=beams, frames=frames)

dataloader = pre_pro.batching(input, output, workers=4)




NameError: name 'torch' is not defined

In [3]:
'Network set up'
network_info = torch.load('nets/MSE/MSE_250_350.pt', weights_only=True)

model_config = (3, 528 ,1024 , 1024, 528, 528)
net = Net(torch.cos, *model_config).to(device)
# net.load_state_dict(network_info['model_state_dict'])
optimizer = torch.optim.Adam(net.parameters(), lr=1e-4)
# optimizer.load_state_dict(network_info['optimizer_state_dict'])
# for param_group in optimizer.param_groups:
#     param_group['lr'] = 1e-4


In [ ]:
'Traing MSE NN'
save_path = 'nets/MSE/MSE_250_350.pt'
trainer = pre_pro.temp_nn_wrap('cuda', dataloader,model_config,optimizer, net )
trainer.MSE_train(10, prints=True, saving=True, save_path= save_path)

Epoch 0, Batch 0, Loss: 3.823828
Epoch 0, Batch 1, Loss: 2.645728
Epoch 0, Batch 2, Loss: 1.862659
Epoch 0, Batch 3, Loss: 1.240825
Epoch 0, Batch 4, Loss: 0.842804
Epoch 0, Batch 5, Loss: 0.694655
Epoch 0, Batch 6, Loss: 0.697706
Epoch 0, Batch 7, Loss: 0.761278
Epoch 0, Batch 8, Loss: 0.836143
Epoch 0, Batch 9, Loss: 0.878005
Epoch 0, Batch 10, Loss: 0.857117
Epoch 0, Batch 11, Loss: 0.791058
Epoch 0, Batch 12, Loss: 0.729873
Epoch 0, Batch 13, Loss: 0.678439
Epoch 0, Batch 14, Loss: 0.682102
Epoch 0, Batch 15, Loss: 0.717581
Epoch 0, Batch 16, Loss: 0.740039
Epoch 0, Batch 17, Loss: 0.764772
Epoch 0, Batch 18, Loss: 0.742212
Epoch 0, Batch 19, Loss: 0.706525
Epoch 0, Batch 20, Loss: 0.698804
Epoch 0, Batch 21, Loss: 0.674551
Epoch 0, Batch 22, Loss: 0.688321
Epoch 0, Batch 23, Loss: 0.712026
Epoch 0, Batch 24, Loss: 0.716464
Epoch 0, Batch 25, Loss: 0.711822
Epoch 0, Batch 26, Loss: 0.699942
Epoch 0, Batch 27, Loss: 0.694062
Epoch 0, Batch 28, Loss: 0.663846
Epoch 0, Batch 29, Loss:

(0.019959298619890913, 9)

In [4]:
'Adding in Regularazation'

network_info = torch.load('nets/MSE/MSE_250_350.pt', weights_only=True)
retrain_net = Net(torch.cos, *network_info['model config'])
retrain_net.load_state_dict(network_info['model_state_dict'])
retrain_net.to('cuda')
spt_optimizer = torch.optim.Adam(retrain_net.parameters(), lr = 0 )
spt_optimizer.load_state_dict(network_info['optimizer_state_dict'])
for param_group in spt_optimizer.param_groups:
    param_group['lr'] = 5e-5

In [ ]:
'New mini-batching'

dataset = pre_pro.FrameDataset(inputs_sorted=input, outputs_sorted= output)
spt_dataloader = DataLoader(dataset, pin_memory=True, num_workers=1, persistent_workers= True, batch_size=1)
it = iter(spt_dataloader)


In [1]:
'Space Time Training-Hyperparameter Tunning'

retrainer = pre_pro.temp_nn_wrap('cuda', spt_dataloader, model_config=network_info['model config'],
                                  optimizer=spt_optimizer, net = retrain_net)

retrainer.SP_T_train(1, prints=True, saving= True, save_path='nets/SPT/spt_250_350.pt', ly =0 )

NameError: name 'pre_pro' is not defined